# LLM API Integration, Token Logging, and Streaming

**Tutorial reference:** Part 2 — Layer 1 of the SmartIntake Learning Tutorial

This notebook covers the foundational layer: making authenticated, controlled API calls to an LLM and logging token usage in a compliance-safe way. Everything else in SmartIntake sits on top of this.

---

**Summary**

- Made an authenticated API call to the Anthropic API
- Inspected the token usage object
- Written a compliance-safe token log to `debug.log`
- Experimented with `max_tokens` to understand its effect
- Compared batch and streaming response delivery

## 1. Setup and Authentication

Store your API key in a `.env` file — never hardcode it in a notebook or source file.

Your `.env` file should contain a single line:
```
ANTHROPIC_API_KEY=sk-ant-...
```

In [ ]:
# Load environment variables from the .env file.
# python-dotenv reads the file and makes the key available via os.getenv().

from dotenv import load_dotenv
import os
import anthropic

load_dotenv()

# Load ANTHROPIC_API_KEY from the project-root .env (one folder up from week1/).
load_dotenv(dotenv_path=os.path.join('..', '.env'))

assert os.environ.get('ANTHROPIC_API_KEY'), (
    'ANTHROPIC_API_KEY not found. Add it to the project-root .env file.'
)

# The client reads ANTHROPIC_API_KEY from the environment automatically.
client = anthropic.Anthropic()

MODEL = 'claude-haiku-4-5'   # supports temperature / top_k / top_p (Opus 4.8 / 4.7 do not)
print(f'Client ready. Using model: {MODEL}')

## 2. A Basic Completion Call

The simplest possible call: send a regulatory query and receive a response.

Notice the two parameters we set deliberately:
- `max_tokens=300` — sufficient for a compact extraction response; prevents runaway generation
- `model` — use `claude-sonnet-4-6`; match the version in your course materials

In [ ]:
# A simple extraction call with no function calling yet.
# We ask the model to return a JSON object directly in its text response.
# Structured output via function calling is introduced in NB-04.

BASIC_SYSTEM = """
You are a pharmaceutical regulatory intake specialist.
Extract the following five fields from the user's message and return them as a JSON object:
query_type, regulation_ref, product_area, urgency, submitting_team.
Return only the JSON object. No additional text.
"""

test_message = (
    "CMC Regulatory here. FDA issued a 483 observation on our "
    "manufacturing site in Pune. We have 15 business days to respond."
)

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=300,        # Deliberately small — extraction does not need more
    system=BASIC_SYSTEM,
    messages=[
        {"role": "user", "content": test_message}
    ]
)

# The response text is in response.content[0].text
print("Model response:")
print(response.content[0].text)

## 3. Inspecting Token Usage

Every API response includes a `usage` object with token counts. This is what we log to `debug.log`.

In [ ]:
# The usage object tells us how many tokens were consumed in this call.
# We log these counts — not the message content.

print("Token usage:")
print(f"  Input tokens:  {response.usage.input_tokens}")
print(f"  Output tokens: {response.usage.output_tokens}")
print(f"  Total:         {response.usage.input_tokens + response.usage.output_tokens}")

## 4. Compliance-Safe Token Logging

We log token usage to `debug.log`. The log entry must contain timestamps and token counts. It must never contain the raw user message.

In [ ]:
import logging
from datetime import datetime
import os

# Create the output directory if it does not exist
os.makedirs("output", exist_ok=True)

# Configure a dedicated logger that writes to debug.log
# Using a FileHandler ensures log entries go to the file, not the notebook output
debug_logger = logging.getLogger("smartintake.debug")
debug_logger.setLevel(logging.DEBUG)

# Remove any existing handlers to avoid duplicate entries when re-running the cell
debug_logger.handlers.clear()

file_handler = logging.FileHandler("debug.log")
file_handler.setFormatter(logging.Formatter("%(asctime)s — %(message)s"))
debug_logger.addHandler(file_handler)


def log_token_usage(event_name: str, input_tokens: int, output_tokens: int):
    """
    Write a compliance-safe token usage entry to debug.log.
    
    COMPLIANCE RULE: This function must never accept or log the raw user message.
    If you find yourself adding a 'user_message' parameter here, stop — that is a violation.
    """
    debug_logger.debug(
        f"event={event_name} input_tokens={input_tokens} output_tokens={output_tokens} "
        f"total={input_tokens + output_tokens}"
    )


# Log the token usage from the previous call
log_token_usage(
    event_name="extraction_call",
    input_tokens=response.usage.input_tokens,
    output_tokens=response.usage.output_tokens,
)

# Verify the log entry was written
with open("debug.log", "r") as f:
    print("Contents of debug.log:")
    print(f.read())

## 5. Experiment — Effect of max_tokens

Try the same extraction call with different `max_tokens` values.

Observe: the extraction output fits comfortably in 300 tokens. Setting a larger value wastes budget and can lead to the model adding explanatory text around the JSON.

In [ ]:
# Run the same extraction call at three different max_tokens settings.
# Compare the output and the token counts.

for max_tok in [100, 300, 800]:
    r = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=max_tok,
        system=BASIC_SYSTEM,
        messages=[{"role": "user", "content": test_message}]
    )
    output = r.content[0].text
    print(f"max_tokens={max_tok}")
    print(f"  Output tokens used: {r.usage.output_tokens}")
    # Truncate long outputs for display
    print(f"  Response: {output[:200]}")
    print()

## 6. Extension — Streaming Responses

Streaming returns tokens incrementally as the model generates them. This is useful for confirmation messages where the user reads every word.

Compare the experience of batch vs streaming output.

In [ ]:
# Streaming call using the Anthropic client's stream context manager.
# Each text delta arrives as soon as it is generated by the model.

CONFIRMATION_PROMPT = """
Generate a short formal confirmation message for this regulatory intake record.
Reference the regulation and query type formally. State the urgency tier. 
End with: 'Please confirm or correct before this record is saved.'
Keep it under 60 words.
"""

intake_record = (
    '{"query_type": "inspection", "regulation_ref": "FDA_21CFR", '
    '"product_area": "cmc", "urgency": "urgent", "submitting_team": "CMC Regulatory"}'
)

print("Streaming confirmation message:")
print("-" * 50)

# The stream() context manager yields text delta events
with client.messages.stream(
    model="claude-sonnet-4-6",
    max_tokens=150,
    system=CONFIRMATION_PROMPT,
    messages=[{"role": "user", "content": f"Intake record: {intake_record}"}]
) as stream:
    for text_chunk in stream.text_stream:
        # print with end="" and flush=True so each chunk appears immediately
        print(text_chunk, end="", flush=True)

print()  # Newline after streaming completes